In [8]:
import numpy as np

#XOR truth table

X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])
Y = np.array([[0],
              [1],
              [1],
              [0]]) #this is the only difference from OR 

In [17]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward(X, weights, bias):
    z = np.dot(X, weights) + bias
    a = sigmoid(z)
    return a

def mse_loss(Y_true, y_pred):
    return np.mean((Y_true - y_pred) ** 2)

def backward(X, Y, a):
    n = Y.shape[0]

    dL_da = (2/n) * (a - Y)
    da_dz = a * (1 - a)
    dz_dz = dL_da * da_dz

    dL_dw = np.dot(X.T, dz_dz)
    dL_db = np.sum(dz_dz)

    return dL_dw, dL_db

np.random.seed(42)

weights = np.random.randn(2, 1)
bias = np.random.randn(1)


In [18]:
learning_rate = 0.1
epochs = 10000

for epoch in range(epochs):
    predictions = forward(X, weights, bias)
    loss = mse_loss(Y, predictions)
    dW, dB = backward(X, Y, predictions)
    
    weights = weights - learning_rate * dW
    bias = bias - learning_rate * dB
    
    if epoch % 1000 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 0, Loss: 0.2916
Epoch 1000, Loss: 0.2500
Epoch 2000, Loss: 0.2500
Epoch 3000, Loss: 0.2500
Epoch 4000, Loss: 0.2500
Epoch 5000, Loss: 0.2500
Epoch 6000, Loss: 0.2500
Epoch 7000, Loss: 0.2500
Epoch 8000, Loss: 0.2500
Epoch 9000, Loss: 0.2500


In [19]:
final_predictions = forward(X, weights, bias)

print("Final predictions:")
print(final_predictions)

print("\nRounded (0 or 1):")
print(np.round(final_predictions))

print("\nActual Y:")
print(Y)

Final predictions:
[[0.5]
 [0.5]
 [0.5]
 [0.5]]

Rounded (0 or 1):
[[1.]
 [1.]
 [1.]
 [0.]]

Actual Y:
[[0]
 [1]
 [1]
 [0]]


## Result: Single Neuron Fails on XOR (As Expected)

**Status:** Confirmed limitation — this is the point

Using the exact same neuron, forward pass, loss, and backward pass that successfully learned OR, training on XOR produced a very different outcome:

- Loss dropped briefly, then **flatlined at exactly 0.2500** from ~epoch 1000 onward — 9000+ additional epochs made zero improvement
- Final predictions converged to **0.5 for every input** — the neuron essentially gave up distinguishing between cases entirely
- This isn't a bug, a learning rate issue, or a training-time issue. It's a hard mathematical ceiling.

### Why This Happens

A single neuron can only draw **one straight decision line** through its input space (geometrically: `weights · X + bias`). Plotting the XOR truth table shows why that's fatal here:

- `(0,0) → 0`
- `(0,1) → 1`
- `(1,0) → 1`
- `(1,1) → 0`

The two `1`s and two `0`s sit **diagonally opposite** each other — no single straight line can separate them. Guessing `0.5` everywhere is mathematically the *best* a lone neuron can do, which is exactly why loss got stuck at 0.25 and refused to move.

### Why This Matters

This is a real, famous result in AI history — proven back in 1969, and a big part of why neural network research stalled for over a decade afterward. We just personally reproduced it from scratch.

**The fix:** stacking neurons into layers. A hidden layer lets the network combine multiple straight lines into a bent decision boundary — enough to correctly separate XOR. This is the actual origin of "deep" in deep learning: depth buys the ability to represent more complex boundaries, not just more capacity.

### Next Up
- [ ] Build a 2-layer network (1 hidden layer) from scratch
- [ ] Re-attempt XOR and confirm it converges properly this time
- [ ] Compare decision boundaries conceptually: 1 line (single neuron) vs. bent boundary (multi-layer)